# 成分广度来源核查：不拟合、不按收益挑特征
验证最新已披露持仓篮子的价格覆盖和时点。固定三个描述量，只检查是否可构建，不分析预测收益。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
card=root/'research/constituent_breadth_source_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/plan['provider'];manifest=json.loads((provider/'qlib_store_manifest.json').read_text())
for name,digest in plan['source_hashes'].items():assert hashlib.sha256((provider/name).read_bytes()).hexdigest()==digest
history=pd.read_csv(provider/'sec_nport_holdings_history.csv',parse_dates=['report_date','filing_date'])
qlib.init(provider_uri=str(provider),region='us')
calendar=pd.DatetimeIndex(D.calendar(start_time=manifest['calendar_start'],end_time=manifest['calendar_end'],freq='day'))
raw=D.features(manifest['instruments'],['$adj_close'],start_time=calendar[0],end_time=calendar[-1],freq='day')
prices=raw['$adj_close'].unstack('instrument').reindex(calendar)
# Do not backfill unavailable symbols or carry a price over a missing native day.
prices=prices.reindex(columns=sorted(history.ticker.unique()))
r5=prices/prices.shift(5)-1.;daily=prices/prices.shift(1)-1.
minimum=prices.rolling(20,min_periods=20).min();new_low=prices.le(minimum)
snapshots=history[['filing_date','report_date','accession']].drop_duplicates().sort_values(['filing_date','accession'])
snapshots=snapshots.drop_duplicates('filing_date',keep='last')
rows=[];missing=[]
for day in calendar:
 eligible=snapshots.loc[snapshots.filing_date.lt(day)]  # date-only filings: never same-day use
 if eligible.empty:continue
 snap=eligible.iloc[-1];members=history.loc[history.accession.eq(snap.accession)].set_index('ticker');weights=members.weight_pct
 valid=prices.loc[day,weights.index].gt(0)&np.isfinite(r5.loc[day,weights.index])&minimum.loc[day,weights.index].notna()&np.isfinite(daily.loc[day,weights.index])
 total=float(weights.sum());covered=float(weights.loc[valid].sum());full=bool(valid.all())
 row={'date':day,'accession':snap.accession,'report_date':snap.report_date,'filing_date':snap.filing_date,'report_age_days':(day-snap.report_date).days,'filing_age_days':(day-snap.filing_date).days,'members':len(weights),'covered_members':int(valid.sum()),'reported_weight':total,'covered_weight':covered,'coverage_fraction':covered/total,'complete_reported_basket':full,'new_low20_share':np.nan,'positive_return5_share':np.nan,'downside_top5_share':np.nan}
 missing_fraction=(total-covered)/total
 for name,condition in [('new_low20',new_low.loc[day,weights.index]),('positive_return5',r5.loc[day,weights.index].gt(0))]:
  lower=float(weights.loc[valid & condition].sum()/total)
  row[name+'_lower']=lower;row[name+'_upper']=min(1.,lower+missing_fraction)
  assert 0<=row[name+'_lower']<=row[name+'_upper']<=1.+1e-12
 for symbol in weights.index[~valid]:missing.append({'date':day,'symbol':symbol,'missing_weight':float(weights[symbol])})
 if full:
  row['new_low20_share']=float(weights[new_low.loc[day,weights.index]].sum()/total)
  row['positive_return5_share']=float(weights[r5.loc[day,weights.index].gt(0)].sum()/total)
  downside=weights*(-daily.loc[day,weights.index]).clip(lower=0)
  row['downside_top5_share']=float(downside.nlargest(5).sum()/downside.sum()) if downside.sum()>0 else np.nan
 rows.append(row)
coverage=pd.DataFrame(rows).set_index('date');missing=pd.DataFrame(missing)
annual=coverage.groupby(coverage.index.year).agg(sessions=('members','size'),complete_basket_sessions=('complete_reported_basket','sum'),minimum_coverage=('coverage_fraction','min'),median_coverage=('coverage_fraction','median'),median_report_age=('report_age_days','median'),maximum_report_age=('report_age_days','max'))
missing_summary=missing.groupby('symbol').agg(missing_sessions=('date','size'),maximum_missing_weight=('missing_weight','max')).sort_values('missing_sessions',ascending=False) if len(missing) else pd.DataFrame()
windows=[]
for name,start,end in [('2023Q4','2023-10-01','2023-12-31'),('2026Q3','2026-07-01','2026-09-30')]:
 import exchange_calendars as xc
 expected=len(xc.get_calendar('XNYS',start=start,end=end).schedule);part=coverage.loc[start:end]
 windows.append({'window':name,'expected_sessions':expected,'observed_sessions':len(part),'complete_reported_basket_sessions':int(part.complete_reported_basket.sum()),'missing_sessions':expected-len(part),'median_breadth_bound_width':float((1-part.coverage_fraction).median()),'maximum_breadth_bound_width':float((1-part.coverage_fraction).max())})
lag=(snapshots.filing_date-snapshots.report_date).dt.days
summary={'experiment':card.name,'decision':'reported_holdings_proxy_only_needs_membership_and_price_completion','new_fits':0,'new_backtests':0,'outcome_correlations':0,'parameter_searches':0,'objective_achieved':False,'provider_uri':str(provider),'region':'us','as_of':manifest['as_of'],'source_first_filing':str(snapshots.filing_date.min().date()),'source_last_filing':str(snapshots.filing_date.max().date()),'filing_lag_days_min':int(lag.min()),'filing_lag_days_median':float(lag.median()),'filing_lag_days_max':int(lag.max()),'observed_sessions':len(coverage),'complete_reported_basket_sessions':int(coverage.complete_reported_basket.sum()),'missing_symbols':list(missing_summary.index),'actual_contemporaneous_index_membership_certified':False,'exact_publication_clock_certified':False,'proxy_role':'Latest previously filed reported equity basket, fixed disclosed weights, not daily SOXX index weights','qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/coverage','analysis/missing','analysis/candidate_features']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='no_fit_constituent_source_review',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,outcome_correlations=0)
 rec.save_objects(artifact_path='inputs',plan=plan,manifest=manifest,holdings=history)
 rec.save_objects(artifact_path='analysis',coverage=coverage,missing=missing,candidate_features=coverage[['new_low20_share','positive_return5_share','downside_top5_share','new_low20_lower','new_low20_upper','positive_return5_lower','positive_return5_upper']],annual=annual,windows=pd.DataFrame(windows),summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),complete_basket_sessions=summary['complete_reported_basket_sessions'])
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json')
annual.to_csv(card/'annual_coverage.csv');missing_summary.to_csv(card/'missing_members.csv');pd.DataFrame(windows).to_csv(card/'window_coverage.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
print(annual.to_string());print(pd.DataFrame(windows).to_string(index=False));print(missing_summary.to_string());print('lag',lag.min(),lag.median(),lag.max())


# 成分修复广度：先发现数据问题，不拟合

本轮0拟合、0回测、0未来收益相关检验、0参数搜索。只对现有N-PORT来源和Qlib价格做时点、成员与覆盖检查，计算固定描述特征及缺失界限，不声称预测有效性。

现有数据是最新已披露的历史持仓篮子，不能当成当日真实SOXX指数成员及权重。披露延迟中位57日；所用报告在交易日通常已陈旧约103–106日。日期级披露缺精确发布时间，本卡排除同日申报使用，且不认证真实首收。未把报告期回填为已知日期。

价格截至2026-07-02。2019-11-26起1657个有披露篮子的交易日，仅398日成员价格完整；2019–2023年没有完整篮子日。缺失涉及WOLF、XLNX、CREE、CCMP、MXIM、BRKS、IPHI，不能删除后假称全体。代码/证券更名、收购或破产身份需单独核对，不直接拼接相似代码。

对差区间的影响：2023Q4共63日都有价格日历，但均缺少部分成员；二元广度的缺失上下界宽度为0.7897个百分点。2026Q3只有2/64日，缺62日，不能用这些数据解释整个2026Q3。

固定描述量：20日新低权重占比、5日上涨权重占比、当日负收益贡献前5名集中度。完整值仅在所有报告成员价格窗口齐全时计算；无负贡献时集中度未知。二元广度另给上下界：已知满足条件权重/报告总权重为下界，再加缺失权重占比为上界。此区间只处理缺价，不覆盖持仓陈旧、指数与基金差异或历史复权修订误差。未使用未来收益挑选特征或填补方式。

另发现并修复明确的报告版本问题：旧函数按最新提交日期选篮子。2026-07-13提交的文件报告期为2025-09-30，可能覆盖已知2026-03-31持仓。修复后先选当时已公开的最新报告期，再选同报告期最新版本；实际7月14日读回保留2026-03-31/2026-05-28版本。7项相关测试通过。此前价格到7月2日，全部1657日原选择未变，原provider和历史模型未重建；本修复不被说成解释了旧亏损。

决定：reported_holdings_proxy_only_needs_membership_and_price_completion。当前只能做披露篮子代理的描述，尚不能认证真实当时成分广度，更没有超额证据。下一步按CUSIP/公司行动核对缺失成员，并补齐差区间真实价格；对2023Q4先使用边界而非单一伪精确值。保持不拟合、不调参、不按收益选择特征。

[年度覆盖](annual_coverage.csv) · [差区间覆盖](window_coverage.csv) · [缺失成员](missing_members.csv) · [报告版本修复](late_report_fix.json)

Recorder `70487e2f47d0486ba4f15745a1389c2a`。
